# Урок 7. Представление целых чисел в памяти

8 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [← Урок 6](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-06.ipynb) · [Урок 8 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-08.ipynb)

---

Разрядная сетка. Беззнаковое представление и его диапазон. Знаковый бит. Дополнительный код и отрицательные числа.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `ИНФОРМАТИКА` — оставь поля
#@markdown пустыми, они заполнятся сами.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 8А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="08-07", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Память конечна

На бумаге число может быть какой угодно длины. В памяти компьютера
под число отводится **фиксированное количество разрядов** — обычно
8, 16, 32 или 64 бита. Эта ячейка называется **разрядной сеткой**.

Отсюда следует всё остальное в этом уроке: раз разрядов конечное число,
то и чисел помещается конечное количество.

### Беззнаковое представление

Самый простой случай: все разряды заняты самим числом, отрицательных
чисел не бывает.

В n разрядах помещается $2^n$ различных комбинаций, значит числа
от 0 до $2^n - 1$:

| Разрядов | Комбинаций | Диапазон |
|---|---|---|
| 8 | 256 | 0 … 255 |
| 16 | 65 536 | 0 … 65 535 |
| 32 | ≈ 4,3 млрд | 0 … 4 294 967 295 |

Число 255 в восьми разрядах — это `11111111`. А 256 уже не помещается:
для него нужен девятый разряд, которого нет.

### Переполнение

Что произойдёт, если к 255 прибавить единицу в восьмиразрядной сетке?

```
    1 1 1 1 1 1 1 1     (255)
  +               1       (1)
  ─────────────────
  1 0 0 0 0 0 0 0 0     (256 — девять разрядов!)
    └───────────────┘
       помещается только это
```

Старший разряд отбрасывается, и в ячейке остаётся `00000000` — ноль.
Это называется **переполнением**.

Переполнение — не абстракция. В 2014 году счётчик просмотров ролика
на YouTube переполнил 32-разрядную ячейку, и разработчикам пришлось
срочно переходить на 64 разряда. Гораздо хуже кончилось переполнение
в ракете Ariane 5 в 1996 году: ошибка преобразования числа
уничтожила аппарат стоимостью 370 миллионов долларов.

### Как записать отрицательное число

Минуса в памяти нет — есть только нули и единицы. Значит, знак нужно
как-то закодировать теми же битами.

**Наивная идея:** отвести старший разряд под знак (0 — плюс, 1 — минус),
остальные — под модуль. Так называемый прямой код.

Идея не прижилась по двум причинам. Во-первых, получается два нуля:
`00000000` и `10000000` — «плюс ноль» и «минус ноль». Во-вторых, сложение
перестаёт работать: `5 + (−5)` даёт не ноль, а какую-то ерунду,
и процессору приходится отдельно разбирать знаки.

### Дополнительный код

Решение, которое используют все современные компьютеры.
Отрицательное число записывают так:

1. взять модуль числа в двоичном виде;
2. **инвертировать** все разряды (нули на единицы, единицы на нули);
3. **прибавить единицу**.

Найдём представление −5 в восьми разрядах:

```
  5             =  0000 0101
  инвертируем   =  1111 1010
  прибавляем 1  =  1111 1011     ← это и есть −5
```

Проверим, что сложение теперь работает:

```
    0000 0101      (5)
  + 1111 1011     (−5)
  ───────────
  1 0000 0000      девятый разряд отбрасывается
    0000 0000      = 0  ✓
```

Вот в чём красота дополнительного кода: **вычитание превращается
в сложение**. Процессору не нужна отдельная схема вычитания —
достаточно сумматора. Именно поэтому на прошлом уроке остался
вопрос «как компьютер вычитает»; ответ — он не вычитает.

### Диапазон со знаком

Старший разряд в дополнительном коде фактически отвечает за знак:
у всех отрицательных чисел он равен 1. Половина комбинаций уходит
на отрицательные:

| Разрядов | Диапазон со знаком |
|---|---|
| 8 | −128 … 127 |
| 16 | −32 768 … 32 767 |
| 32 | −2 147 483 648 … 2 147 483 647 |

Обратите внимание на асимметрию: отрицательных чисел на одно больше.
Нулю нужна ровно одна комбинация (двух нулей больше нет), и освободившаяся
комбинация ушла к отрицательным. Число −128 в восьми разрядах есть,
а +128 — уже нет.

Общая формула диапазона: от $-2^{n-1}$ до $2^{n-1} - 1$.

## Смотрим, как это работает

### Пример 1. Границы разрядной сетки

In [ ]:
for разрядов in [8, 16, 32]:
    без_знака = 2 ** разрядов - 1
    от = -2 ** (разрядов - 1)
    до = 2 ** (разрядов - 1) - 1
    print(f"{разрядов:>2} разрядов: без знака 0…{без_знака:<12} со знаком {от}…{до}")

Обратите внимание, как быстро растут границы: каждые дополнительные
восемь разрядов увеличивают диапазон в 256 раз.

### Пример 2. Дополнительный код по шагам

In [ ]:
def дополнительный_код(число, разрядов=8):
    if число >= 0:
        return f"{число:0{разрядов}b}"

    модуль = f"{abs(число):0{разрядов}b}"
    инверсия = ""
    for б in модуль:
        if б == "0":
            инверсия += "1"
        else:
            инверсия += "0"
    результат = f"{int(инверсия, 2) + 1:0{разрядов}b}"

    print(f"  модуль {abs(число)}   = {модуль}")
    print(f"  инверсия      = {инверсия}")
    print(f"  плюс единица  = {результат}")
    return результат


print("Представление -5:")
код = дополнительный_код(-5)
print("Ответ:", код)
print()
print("Представление  5:", дополнительный_код(5))

Строка `f"{число:08b}"` даёт двоичную запись, дополненную нулями
до восьми разрядов. Запись `0{разрядов}b` подставляет нужную ширину
из переменной — так одна функция работает для любой разрядной сетки.

### Пример 3. Переполнение своими глазами

Смоделируем восьмиразрядную ячейку и посмотрим, что происходит
на границе.

In [ ]:
def в_ячейку(число, разрядов=8):
    """Что окажется в ячейке, если положить туда число (без знака)."""
    return число % (2 ** разрядов)


for число in [254, 255, 256, 257, 511, 512]:
    print(f"  кладём {число:>3} → в ячейке {в_ячейку(число):>3}")

Операция «остаток от деления на 256» — это и есть отбрасывание лишних
старших разрядов. Число 256 превращается в 0, 257 — в 1, и счёт идёт
по кругу. Такую арифметику называют **кольцевой**, и она в точности
описывает поведение реального процессора.

## Пробуем сами

### Задача 1. Диапазон без знака

По количеству разрядов верните список из двух чисел: минимальное
и максимальное значение **беззнакового** целого.

`диапазон_без_знака(8)` → `[0, 255]`

In [ ]:
def диапазон_без_знака(разрядов):
    return ...

In [ ]:
si.check("1", диапазон_без_знака, [
    (8, [0, 255]),
    (16, [0, 65535]),
    (1, [0, 1]),
    (4, [0, 15]),
])

### Задача 2. Диапазон со знаком

То же самое, но для чисел со знаком в дополнительном коде.
Не забудьте про асимметрию.

`диапазон_со_знаком(8)` → `[-128, 127]`

In [ ]:
def диапазон_со_знаком(разрядов):
    return ...

In [ ]:
si.check("2", диапазон_со_знаком, [
    (8, [-128, 127]),
    (16, [-32768, 32767]),
    (4, [-8, 7]),
])

### Задача 3. Наименьшее восьмиразрядное

Какое наименьшее целое число можно записать в восьмиразрядной ячейке
в дополнительном коде? Впишите ответ числом.

In [ ]:
ответ = 0

si.check_value("3", ответ, "55e680185552dd73",
               hint="Половина из 256 комбинаций уходит на отрицательные.")

## Домашнее задание

### Домашнее задание 1. Дополнительный код

Напишите функцию, которая возвращает представление целого числа
в дополнительном коде в заданном числе разрядов — строкой.

Для неотрицательных чисел это просто двоичная запись с ведущими нулями.
Для отрицательных — инверсия и плюс единица, как в примере 2.

Существует и короткий путь: представление числа −x в n разрядах
совпадает с записью числа $2^n - x$. Убедитесь в этом сами
на примере −5 в восьми разрядах.

In [ ]:
def в_доп_код(число, разрядов):
    return ...

In [ ]:
si.check("дз1", в_доп_код, [
    ((5, 8), "00000101"),
    ((-5, 8), "11111011"),
    ((0, 8), "00000000"),
    ((-1, 8), "11111111"),
    ((-128, 8), "10000000"),
    ((127, 8), "01111111"),
])

### Домашнее задание 2. Обратное чтение

Напишите функцию, которая читает двоичную запись как число
**со знаком** в дополнительном коде.

Правило простое: если старший разряд равен нулю — число положительное,
читается как обычно. Если единица — вычтите из значения $2^n$,
где n — количество разрядов.

`из_доп_кода("11111011")` → `-5`

In [ ]:
def из_доп_кода(запись):
    return ...

In [ ]:
si.check("дз2", из_доп_кода, [
    ("11111011", -5),
    ("00000101", 5),
    ("11111111", -1),
    ("10000000", -128),
    ("01111111", 127),
    ("0000", 0),
])

### Домашнее задание 3. Сложение с переполнением

Напишите функцию, которая складывает два числа в разрядной сетке
заданного размера **без знака** и возвращает то, что реально окажется
в ячейке.

`сложить_в_ячейке(255, 1, 8)` → `0`, потому что произошло переполнение.

Дополнительно верните признак переполнения: функция должна возвращать
список из двух элементов — `[результат, было_ли_переполнение]`.

In [ ]:
def сложить_в_ячейке(а, б, разрядов):
    return ...

In [ ]:
si.check("дз3", сложить_в_ячейке, [
    ((255, 1, 8), [0, True]),
    ((100, 100, 8), [200, False]),
    ((200, 100, 8), [44, True]),
    ((0, 0, 8), [0, False]),
    ((65535, 1, 16), [0, True]),
])

---

### Проверьте на настоящем железе

Python сам умеет работать с числами любой длины — он выделяет память
по мере надобности, поэтому переполнения в нём не бывает. Но большинство
языков ведут себя иначе, и в них `255 + 1` в байтовой переменной честно
даст ноль.

Попробуйте объяснить своими словами, почему в Python `2 ** 1000`
вычисляется без ошибок, а в калькуляторе телефона — нет.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 6](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-06.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 8 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-08.ipynb)